# Anomaly detection on sensor time series

Four detectors on four labeled series from the [Numenta Anomaly Benchmark](https://github.com/numenta/NAB):
rolling z-score (baseline), Isolation Forest on window features, matrix profile discords (stumpy) and a 1D-conv autoencoder.

Each series is split chronologically into train (40%), validation (20%) and test (40%). Detectors are fit on train with labeled windows removed,
the threshold is the same score quantile on normal validation points for every detector, and evaluation runs on test at event level:
a labeled window counts as detected if at least one alarm falls inside it, each alarm run outside a window is one false positive.
Delay is measured from the labeled anomaly point, so negative values mean early alarms.

In [ ]:
import json

import pandas as pd
import plotly.io as pio

from src.data import ROOT
from src.plots import hero, small_multiples

pio.renderers.default = "notebook_connected"
RESULTS = ROOT / "results"
metrics = pd.read_csv(RESULTS / "metrics.csv")
scores = pd.read_parquet(RESULTS / "scores.parquet")
meta = json.loads((RESULTS / "meta.json").read_text())

## Results

In [ ]:
table = metrics.pivot(index="series", columns="detector", values=["f1", "delay_h", "point_f1"])
table.style.format("{:.2f}", na_rep="–")

In [ ]:
metrics.groupby("detector")[["precision", "recall", "f1", "delay_h", "point_f1"]].mean().round(2)

## Detections over time

Grey bands are the labeled anomaly windows, the dotted line marks the start of the test part.

In [ ]:
fig = hero(scores, meta, "machine_temperature")
fig.write_image(ROOT / "docs" / "hero.png", width=1400, height=480, scale=2)
fig

In [ ]:
for name in meta:
    small_multiples(scores, meta, name).show()